# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoonakhan/flyrank-ml-internship-task1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (one pseudonymized page), aggregated over a trailing 90-day window.**

The dataset is a snapshot — not daily records. Every metric in the row (impressions, clicks, sessions, etc.) is summed or averaged across the most recent 90 days ending at the export date. The 30-day comparison columns (`impressions_last_30d`, `impressions_prev_30d`) are nested inside that same 90-day window. There is no future data in this slice; the label is drawn from the current window only.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Unique client_id: {df['client_id'].nunique()}")
print(f"All content_id unique? {df['content_id'].nunique() == len(df)}")
print("\nGrain confirmed: one row = one content item.")
print("Time window: all *_90d columns are trailing 90-day aggregates.")

Rows: 30,000
Unique content_id: 30,000
Unique client_id: 32
All content_id unique? True

Grain confirmed: one row = one content item.
Time window: all *_90d columns are trailing 90-day aggregates.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|--------|--------|-----|
| **Features** | `impressions_90d`, `clicks_90d`, `sessions_90d`, `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier` | All are observable signals known at the time of scoring. The starter pipeline also uses log-transformed versions (`log_impressions_90d`, etc.) and flags (`has_clicks`, `measurable_opportunity`) which are derived safely from these. |
| **Label** | `is_declining_label` | Binary target: 1 if `trend_direction == "down"`, else 0. This is what the model learns to predict. |
| **Context** | `content_id`, `client_id`, `provider_used`, `model_used` | IDs are for grouping and client-holdout splits only — they carry no predictive signal. `provider_used` and `model_used` are LLM metadata; they describe how the content was made, not how it performs, and the data dictionary marks them as not for modeling. |
| **Excluded** | `trend_direction`, `trend_pct` | **Leakage.** These columns literally define the label. Using them as features would mean the model sees the answer while training. The data dictionary explicitly flags both as "Label source — never a feature." |

In [5]:
# Confirm the excluded fields exist
excluded = ['trend_direction', 'trend_pct']
for col in excluded:
    print(f"{col}: exists = {col in df.columns}")

# Build the label the same way the pipeline does (this is NOT in the raw CSV)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Show that trend_direction perfectly predicts the label (by definition)
label_from_trend = (df['trend_direction'] == 'down').astype(int)
matches = (label_from_trend == df['is_declining_label']).all()
print(f"\ntrend_direction == 'down' perfectly matches is_declining_label: {matches}")
print("This confirms: trend_direction is the label source and must be excluded from features.")

trend_direction: exists = True
trend_pct: exists = True

trend_direction == 'down' perfectly matches is_declining_label: True
This confirms: trend_direction is the label source and must be excluded from features.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify four contract claims below:
1. **Grain:** 30,000 rows, each with a unique `content_id`.
2. **Client distribution:** 32 clients, but counts are unbalanced.
3. **Missing values:** Systematic — keyword columns are blank for `feedly article` rows; `word_count` and `char_count` are blank for 7,699 rows.
4. **Window:** All `*_90d` columns are non-negative aggregates; the 30-day splits are subsets of the 90-day window.

In [6]:
import pandas as pd
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 1. Grain
print("=== GRAIN ===")
print(f"Total rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"One row = one content item? {len(df) == df['content_id'].nunique()}")

# 2. Counts per client (unbalanced)
print("\n=== COUNTS PER CLIENT ===")
client_counts = df['client_id'].value_counts()
print(f"Clients: {df['client_id'].nunique()}")
print(f"Largest client: {client_counts.iloc[0]:,} rows")
print(f"Smallest client: {client_counts.iloc[-1]:,} rows")
print(f"Median client size: {client_counts.median():.0f} rows")

# 3. Missing values
print("\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

# 4. Window sanity check
print("\n=== WINDOW CHECK ===")
print(f"impressions_90d >= impressions_last_30d + impressions_prev_30d?")
# Note: last_30d + prev_30d should roughly equal 90d, but not exactly because
# the 90d window may include more than two 30-day chunks
check = (df['impressions_90d'] >= df['impressions_last_30d'].fillna(0) + df['impressions_prev_30d'].fillna(0)).mean()
print(f"True for {check:.1%} of rows (expected ~100%, since last_30d + prev_30d are subsets of 90d)")

=== GRAIN ===
Total rows: 30,000
Unique content_id: 30,000
One row = one content item? True

=== COUNTS PER CLIENT ===
Clients: 32
Largest client: 7,008 rows
Smallest client: 3 rows
Median client size: 567 rows

=== MISSING VALUES ===
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

=== WINDOW CHECK ===
impressions_90d >= impressions_last_30d + impressions_prev_30d?
True for 100.0% of rows (expected ~100%, since last_30d + prev_30d are subsets of 90d)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell me:**

1. **Causality.** It is observational, not experimental. I can see that stale pages often decline, but I cannot prove that refreshing them *causes* recovery. There is no A/B test or control group.

2. **The future.** The label is drawn from the same 90-day window as the features. It tells me what *already looks* declining, not what *will* decline next month. A true prediction would need a future-window label (features from days 1–90, outcome from days 91–120).

3. **Real identities.** All URLs, queries, titles, domains, and client names are pseudonymized or removed. I cannot look up a page on Google to verify its real performance.

4. **Pre-window history.** The dataset has no data before the trailing 90 days. I cannot check seasonality, long-term trends, or whether a page was previously refreshed.

5. **Why a page declined.** I see signals (lower impressions, lower CTR), but not the reason (algorithm update, competitor content, seasonality, technical error, etc.).

6. **Post-refresh outcomes.** There is no "after refresh" measurement. I cannot validate whether acting on a recommendation actually helped.

In [7]:
# Verify we have no future data and no real identifiers
has_url = any('url' in c and 'hash' not in c for c in df.columns)
has_domain = any('domain' in c for c in df.columns)
has_title = any('title' in c for c in df.columns)
has_future = any('future' in c or 'next_30d' in c for c in df.columns)

print("Data limit checks:")
print(f"  Raw URL columns? {has_url}")
print(f"  Domain columns? {has_domain}")
print(f"  Title columns? {has_title}")
print(f"  Future-window columns? {has_future}")
print("\nConfirmed: no real identifiers, no future data. The dataset is a safe but limited snapshot.")

Data limit checks:
  Raw URL columns? False
  Domain columns? False
  Title columns? False
  Future-window columns? False

Confirmed: no real identifiers, no future data. The dataset is a safe but limited snapshot.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.